In [36]:
#pip install pyspark matplotlib seaborn ipywidgets sklearn

## Anomaly Detection

### Data set up

In [37]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Seattle911").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")



file_path = "/Users/maggiekettle/Desktop/6242 Project/calldata_20251019_processed_v4.csv" 
df_spark = spark.read.csv(file_path, header=True, inferSchema=True)

#print(df_spark.columns)
#print(df_spark.count())

In [38]:
from pyspark.sql import functions as F

df_spark = df_spark.withColumn("event_time", F.to_timestamp("cad_event_original_time_queued", "MM/dd/yyyy hh:mm:ss a"))

df_spark = df_spark.withColumn("date", F.to_date("event_time"))
df_spark = df_spark.withColumn("hour", F.hour("event_time"))

#df_spark.select("cad_event_original_time_queued", "event_time", "date", "hour", "call_type", "call_sign_total_service_time_s").show(10, truncate=False)


In [39]:
hourly_calls = (df_spark.groupBy("date", "hour", "call_type").agg( F.count("*").alias("total_calls"), 
                        F.avg("call_sign_total_service_time_s").alias("avg_service_time")).orderBy("date", "hour"))
#hourly_calls.show(10)

### Response Time Anomalies by Call Type + Time of Day

Detect unusually long or short response times relative to expected norms for that type of call, time of day, and dispatch area.

Use isolation forest to find above 99th or below 1st percnetile

In [40]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Maybe make a rolling z-score over time for finer detection?
window = Window.partitionBy("call_type")

hourly_calls = (hourly_calls.withColumn("mean_calls", F.avg("total_calls").over(window))
                .withColumn("std_calls", F.stddev("total_calls").over(window)))

hourly_calls = hourly_calls.withColumn("z_score",(F.col("total_calls") - F.col("mean_calls")) / (F.col("std_calls") + F.lit(1e-6)))

hourly_calls = hourly_calls.withColumn("is_anomaly", (F.abs(F.col("z_score")) > 2.5).cast("int"))

#hourly_calls.select( "date", "hour", "call_type", "total_calls", "mean_calls", "std_calls", "z_score", "is_anomaly")\
#                        .orderBy("date", "hour").show(20, truncate=False)


In [71]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

hourly_calls_pd = hourly_calls.toPandas()
hourly_calls_pd["datetime"] = pd.to_datetime(hourly_calls_pd["date"]) + pd.to_timedelta(hourly_calls_pd["hour"], unit="h")

#plt.figure(figsize=(14, 6))

#sns.scatterplot(data=hourly_calls_pd, x="datetime", y="total_calls", hue="call_type", style=hourly_calls_pd["is_anomaly"].map({1: "Anomaly", 0: "Normal"}), palette="tab10", alpha=0.8)

#plt.title("Incident Flow Timeline — Hourly 911 Calls with Anomalies Highlighted", fontsize=14)
#plt.xlabel("Date / Hour")
#plt.ylabel("Number of Calls per Hour")
#plt.legend(title="Call Type / Anomaly", bbox_to_anchor=(1.05, 1), loc="upper left")
#plt.tight_layout()
#plt.savefig("hourly_911_response_time_anomalies.png", dpi=300, bbox_inches="tight")
#plt.show()


Most hours show consistent 911 activity, with call volumes typically ranging between 10–35 per hour across all types. The majority of points fall within this normal range, indicating stable response patterns over time. A few clear spikes stand out — moments where calls per hour jump above 60 — suggesting unusual bursts in activity likely tied to large incidents or surges in demand. These anomalies mainly occur in the higher-volume call types like 911 and ONVIEW, while low-volume types remain steady, pointing to system-wide reliability with occasional high-stress periods worth investigating further.

In [42]:
hourly_calls_w_cad = (df_spark.select("cad_event_number", "date", "hour", "call_type", "call_sign_total_service_time_s")
                      .join(hourly_calls, on=["date", "hour", "call_type"], how="left"))
#hourly_calls_w_cad.select("cad_event_number", "date", "hour", "call_type","total_calls", "avg_service_time", "is_anomaly").show(10, truncate=False)

### Call Type Mismatch Anomalies (Initial vs Final)

Find cases where the initial classification differs significantly from the final (e.g., “Aid Response” → “Structure Fire”).

Build a confusion matrix or frequency table of transitions; use rare transitions as anomalies.

In [43]:
#set up 
from pyspark.sql import functions as F
INITIAL_COL = "initial_call_type_mapping"
FINAL_COL = "final_call_type_mapping"
#df_spark.select(INITIAL_COL, FINAL_COL).show(5)

In [44]:
transition_counts = (df_spark.groupBy(INITIAL_COL, FINAL_COL).agg(F.count("*").alias("count")).orderBy(F.desc("count")))
#transition_counts.show(20, truncate=False)

In [45]:
total_calls = df_spark.count()

transition_counts = transition_counts.withColumn("pct", F.col("count") / F.lit(total_calls))
transition_counts = transition_counts.withColumn("is_anomaly",(F.col("pct") < 0.005).cast("int"))
#transition_counts.orderBy("pct").show(20, truncate=False)

In [46]:
transition_counts = transition_counts.withColumn("is_self_transition", (F.col(INITIAL_COL) == F.col(FINAL_COL)).cast("int"))

In [72]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

transition_pd = transition_counts.toPandas()
if "is_self_transition" in transition_pd.columns:
    transition_pd = transition_pd[transition_pd["is_self_transition"] == 0]
transition_pd["transition_label"] = transition_pd[INITIAL_COL] + " → " + transition_pd[FINAL_COL]
top_transitions = transition_pd.sort_values("pct", ascending=False).head(15)


#plt.figure(figsize=(10, 6))
#sns.barplot(data=top_transitions, x="pct", y="transition_label", hue="is_anomaly", dodge=False, palette={0: "gray", 1: "red"})
#plt.title("Top Call Type Transitions — Rare Mismatches Highlighted", fontsize=14)
#plt.xlabel("Proportion of Calls")
#plt.ylabel("Initial → Final Call Type")
#plt.legend(title="Anomaly", bbox_to_anchor=(1.05, 1), loc="upper left")
#plt.tight_layout()
#plt.show()


In [73]:
rare_transitions = transition_pd[transition_pd["is_anomaly"] == 1]
#sns.barplot(
#    data=rare_transitions.sort_values("pct", ascending=False).head(15),
#    x="pct", y="transition_label", color="red"
#)
#plt.savefig("hourly_911_call_typeanomalies.png", dpi=300, bbox_inches="tight")
#plt.title("Rare Call Type Mismatches (<0.5% Frequency)")


Most of the common transitions make sense and are just different ways of labeling the same kind of call — like small naming tweaks rather than real mismatches.
In the rare group (<0.5%), almost everything is still just wording differences, but there are a few (around four or five) that look like real anomalies where the type of situation actually changed once responders got there.

In [49]:
CAD_COL = "cad_event_number"

transition_w_cad = (df_spark.select(CAD_COL, INITIAL_COL, FINAL_COL).join(transition_counts, on=[INITIAL_COL, FINAL_COL], how="left"))
#transition_w_cad.select(CAD_COL, INITIAL_COL, FINAL_COL, "count", "pct", "is_anomaly").show(10, truncate=False)

### Geographic Outlier Detection (Spatial Anomalies)

What it measures:

Whether a call’s location (latitude/longitude) is unusually far from where similar calls or calls from that category normally occur.

Goal:

Catch weirdly placed or mis-geocoded calls — i.e., locations that don’t make sense for the type of incident or sector.

“Is this call in a place that’s weird or unexpected for this type of incident?”

In [50]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

LAT_COL = "dispatch_latitude"
LON_COL = "dispatch_longitude"
GROUP_COL = "dispatch_sector"   # could also use "call_type" or "dispatch_precinct"

geo_stats = (df_spark.groupBy(GROUP_COL).agg(
        F.avg(LAT_COL).alias("mean_lat"),
        F.avg(LON_COL).alias("mean_lon"),
        F.stddev(LAT_COL).alias("std_lat"),
        F.stddev(LON_COL).alias("std_lon"),
        F.count("*").alias("n_calls")))

df_geo = df_spark.join(geo_stats, on=GROUP_COL, how="left")

df_geo = (df_geo.withColumn("z_lat", (F.col(LAT_COL) - F.col("mean_lat")) / (F.col("std_lat") + F.lit(1e-6)))
        .withColumn("z_lon", (F.col(LON_COL) - F.col("mean_lon")) / (F.col("std_lon") + F.lit(1e-6))))

df_geo = df_geo.withColumn("spatial_score", F.sqrt(F.col("z_lat")**2 + F.col("z_lon")**2))
df_geo = df_geo.withColumn("is_geo_anomaly", (F.col("spatial_score") > 3).cast("int")) # make it stricter go higher, make it more lenient make it lower
#df_geo.select(GROUP_COL, LAT_COL, LON_COL, "spatial_score", "is_geo_anomaly").show(10, truncate=False)


In [74]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#had to limit to vis in here
geo_pd = (
    df_geo
    .select(GROUP_COL, LAT_COL, LON_COL, "is_geo_anomaly")
    .dropna(subset=[LAT_COL, LON_COL])  
    .sample(fraction=0.1, seed=42)     
    .limit(10000)                       
    .toPandas()
)


#plt.figure(figsize=(8, 6))
#sns.scatterplot(
#    data=geo_pd,
#    x=LON_COL, y=LAT_COL,
#    hue="is_geo_anomaly",
#    palette={0: "gray", 1: "red"},
#    alpha=0.7
#)
#plt.title("Geographic Outlier Detection — Red = Spatial Anomaly", fontsize=14)
#plt.xlabel("Longitude")
#plt.ylabel("Latitude")
#plt.legend(title="Anomaly")
#plt.tight_layout()
#plt.savefig("hourly_911_geographic_anomalies.png", dpi=300, bbox_inches="tight")
#plt.show()


Most call locations fall along one clear geographic line, showing that incidents are concentrated in a consistent service area. Only a few points are flagged as spatial anomalies, and they sit just outside that main corridor. These likely come from minor coordinate errors or calls happening near boundary edges rather than true location mistakes. Overall, the data shows strong geographic consistency with only a few scattered outliers.

### Temporal Burst Detection

Detect sudden surges in call volume in a neighborhood or beat (possible real-world incidents).

Apply moving-window statistics or a Poisson-based anomaly model on counts per time interval

In [52]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


df_spark = df_spark.withColumn("event_hour", F.date_trunc("hour", F.col("event_time")))

hourly_counts = (df_spark.groupBy("event_hour", "call_type").agg(F.count("*").alias("total_calls"))
                 .orderBy("event_hour"))

#hourly_counts.show(10, truncate=False)


In [53]:
w = Window.orderBy("event_hour").rowsBetween(-6, 6)

hourly_counts = (hourly_counts.withColumn("rolling_mean", F.avg("total_calls").over(w)).withColumn("rolling_std", F.stddev("total_calls").over(w)))
hourly_counts = hourly_counts.withColumn("z_score",(F.col("total_calls") - F.col("rolling_mean")) / (F.col("rolling_std") + F.lit(1e-6)))
hourly_counts = hourly_counts.withColumn("is_burst", (F.col("z_score") > 2.5).cast("int"))
#hourly_counts.select("event_hour", "total_calls", "rolling_mean", "z_score", "is_burst").show(10, truncate=False)


In [75]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

hourly_pd = hourly_counts.toPandas()

#plt.figure(figsize=(14,6))
#sns.lineplot(data=hourly_pd, x="event_hour", y="total_calls", label="Total Calls", color="gray")
#sns.scatterplot(
#    data=hourly_pd[hourly_pd["is_burst"] == 1],
#    x="event_hour", y="total_calls",
#    color="red", label="Burst Anomaly", s=60
#)
#plt.title("Temporal Burst Detection — Sudden Increases in 911 Call Volume", fontsize=14)
#plt.xlabel("Time")
#plt.ylabel("Number of Calls per Hour")
#plt.legend()
#plt.tight_layout()
#plt.savefig("hourly_911_temporal_burst_anomalies.png", dpi=300, bbox_inches="tight")
#plt.show()


Now do it per dispatch_sector

In [55]:
sector_window = Window.partitionBy("dispatch_sector").orderBy("event_hour").rowsBetween(-6, 6)

sector_bursts = (df_spark
    .withColumn("event_hour", F.date_trunc("hour", F.col("event_time")))
    .groupBy("dispatch_sector", "event_hour")
    .agg(F.count("*").alias("total_calls"))
    .withColumn("rolling_mean", F.avg("total_calls").over(sector_window))
    .withColumn("rolling_std", F.stddev("total_calls").over(sector_window))
    .withColumn("z_score", (F.col("total_calls") - F.col("rolling_mean")) / (F.col("rolling_std") + F.lit(1e-6)))
    .withColumn("is_burst", (F.col("z_score") > 2.5).cast("int")))


In [76]:
sector_bursts_pd = sector_bursts.toPandas()

#plt.figure(figsize=(14,6))
#sns.lineplot(data=sector_bursts_pd, x="event_hour", y="total_calls", label="Total Calls", color="gray")
#sns.scatterplot(
#    data=sector_bursts_pd[sector_bursts_pd["is_burst"] == 1],
#    x="event_hour", y="total_calls",
#    color="red", label="Burst Anomaly", s=60
#)
#plt.title("Temporal Burst Detection — Sudden Increases in 911 Call Volume per Sector", fontsize=14)
#plt.xlabel("Time")
#plt.ylabel("Number of Calls per Hour")
#plt.legend()
#plt.tight_layout()
#plt.savefig("hourly_911_temporal_burst_anomalies_2.png", dpi=300, bbox_inches="tight")
#plt.show()

At the sector level, small bursts appear regularly, reflecting local fluctuations in activity. However, when aggregated across the entire city, only a handful of major spikes remain, indicating that most surges are localized rather than system-wide. Overall call patterns are steady, with a few short periods of exceptionally high volume that stand out as true temporal anomalies worth further review.

In [57]:
df_w_cad_hourly = (df_spark.select("cad_event_number", "event_hour") .join(hourly_counts, on="event_hour", how="left"))
#df_w_cad_hourly.select("cad_event_number","event_hour","total_calls","rolling_mean","z_score","is_burst").show(10, truncate=False)

df_w_cad_sector = (df_spark.select("cad_event_number", "event_hour", "dispatch_sector").join(sector_bursts, on=["event_hour", "dispatch_sector"], how="left"))
#df_w_cad_sector.select("cad_event_number","dispatch_sector","event_hour","total_calls","z_score","is_burst").show(10, truncate=False)

### Dispatch Routing Anomalies

What it measures:

Whether a dispatch assignment (which station or sector handled the call) makes sense given where the call occurred.

Goal:

Find mismatched or inefficient routing — i.e., when a call was sent to a unit or sector that’s far away from the call location.

“Did the system send the right team to handle this call, based on its location?”

In [58]:
from pyspark.sql import functions as F

def haversine_distance(lat1, lon1, lat2, lon2):
    return (F.lit(6371) * 2 * F.asin(
            F.sqrt(F.pow(F.sin((F.radians(lat2 - lat1)) / 2), 2) + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin((F.radians(lon2 - lon1)) / 2), 2))))


In [59]:
# Group by dispatch sector (or use 'dispatch_beat' or 'dispatch_precinct')
sector_centroids = (df_spark.groupBy("dispatch_sector")
    .agg(F.avg("dispatch_latitude").alias("sector_lat"), F.avg("dispatch_longitude").alias("sector_lon"), F.count("*").alias("n_calls")))


In [60]:
df_dist = df_spark.join(sector_centroids, on="dispatch_sector", how="left")

df_dist = df_dist.withColumn("distance_km",
    haversine_distance( F.col("dispatch_latitude"), F.col("dispatch_longitude"), F.col("sector_lat"), F.col("sector_lon")))


In [61]:
from pyspark.sql.window import Window

w = Window.partitionBy("dispatch_sector")

df_dist = (df_dist
    .withColumn("mean_dist", F.avg("distance_km").over(w))
    .withColumn("std_dist", F.stddev("distance_km").over(w))
    .withColumn("z_score", (F.col("distance_km") - F.col("mean_dist")) / (F.col("std_dist") + F.lit(1e-6)))
    .withColumn("is_routing_anomaly", (F.col("z_score") > 3).cast("int")))

#df_dist.select("dispatch_sector", "dispatch_latitude", "dispatch_longitude", "distance_km", "z_score", "is_routing_anomaly").show(10, truncate=False)

In [77]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

routing_pd = (df_dist.select("dispatch_latitude", "dispatch_longitude", "is_routing_anomaly").dropna(subset=["dispatch_latitude", "dispatch_longitude"])
                     .sample(fraction=0.05, seed=42).toPandas())

#plt.figure(figsize=(8, 6))
#sns.scatterplot(
#    data=routing_pd,
#    x="dispatch_longitude", y="dispatch_latitude",
#    hue="is_routing_anomaly",
#    palette={0: "gray", 1: "red"},
#    alpha=0.7
#)
#plt.title("Dispatch Routing Anomalies — Red = Calls Far from Assigned Sector", fontsize=14)
#plt.xlabel("Longitude")
#plt.ylabel("Latitude")
#plt.legend(title="Anomaly")
#plt.tight_layout()
#plt.savefig("hourly_911_dispatch_routing_anomalies.png", dpi=300, bbox_inches="tight")
#plt.show()


Most calls were routed correctly and fall within expected geographic areas for their sectors. Only a couple of points were flagged as anomalies, sitting at the outer edges of the coverage zone. These few cases likely represent minor boundary overlaps or data entry errors rather than true routing problems, showing overall strong dispatch accuracy.

### Summarize / Putting all results together

In [63]:
print("Response Time Anomalies DF")
#hourly_calls_w_cad.show(5)
print("Call Type Mismatch Anomalies DF")
#transition_w_cad.show(5)
print("Georgraphic Outlier Detection DF")
#df_geo.show(5)
print("Temporal Burst Detection DF")
#df_w_cad_hourly.show(5)
print("per sector")
#df_w_cad_sector.show(5)
print("Dispatch Routing Anomalies")
#df_dist.show(5)

Response Time Anomalies DF
Call Type Mismatch Anomalies DF
Georgraphic Outlier Detection DF
Temporal Burst Detection DF
per sector
Dispatch Routing Anomalies


In [78]:
from pyspark.sql import functions as F

df_response = hourly_calls_w_cad.select("cad_event_number", "is_anomaly")
df_geo_flag = df_geo.select("cad_event_number", "is_geo_anomaly")
df_routing = df_dist.select("cad_event_number", "is_routing_anomaly")
df_burst = df_w_cad_hourly.select("cad_event_number", "is_burst") 
df_type = transition_w_cad.select("cad_event_number", "is_anomaly").withColumnRenamed("is_anomaly", "is_type_anomaly")

df_response = df_response.withColumnRenamed("is_anomaly", "is_response_anomaly")
df_burst = df_burst.withColumnRenamed("is_burst", "is_burst_anomaly")

df_final_anomaly_detection = (
    df_spark
    .select("cad_event_number", "dispatch_sector", "call_type", "event_time")
    .join(df_response, on="cad_event_number", how="left")
    .join(df_burst, on="cad_event_number", how="left")
    .join(df_type, on="cad_event_number", how="left")
    .join(df_geo_flag, on="cad_event_number", how="left")
    .join(df_routing, on="cad_event_number", how="left")
)

df_final_anomaly_detection = df_final_anomaly_detection.withColumn(
    "is_any_anomaly",
    (
        F.col("is_response_anomaly") +
        F.col("is_burst_anomaly") +
        F.col("is_type_anomaly") +
        F.col("is_geo_anomaly") +
        F.col("is_routing_anomaly")
    ).cast("int")
)


#df_final_anomaly_detection.select(
#    "cad_event_number",
#    "is_response_anomaly",
#    "is_burst_anomaly",
#    "is_type_anomaly",
#    "is_geo_anomaly",
#    "is_routing_anomaly",
#    "is_any_anomaly"
#).show(10, truncate=False)


is_response_anomaly → Flags calls that occurred during hours where response times or call volumes were unusually high or low compared to typical patterns for that call type.

is_burst_anomaly → Marks time periods where there was a sudden short-term spike in total 911 call volume — a temporal “burst” indicating potential system overload or major incidents.

is_type_anomaly → Indicates calls where the initial classification of the incident didn’t match the final call type, highlighting possible mislabeling or reclassification during dispatch.

is_geo_anomaly → Flags calls whose location coordinates are unusually far from where similar calls or that dispatch sector typically occur — potential geographic outliers or coordinate errors.

is_routing_anomaly → Identifies calls that were dispatched to a sector or unit far from the incident’s actual location, suggesting routing mismatches or boundary assignment issues.

is_any_anomaly → A combined indicator showing whether a call was flagged by any of the five anomaly detection methods above (1 = at least one anomaly detected).

### Validating

**Valdiating Response Time Anomalies**

In [79]:
from pyspark.sql import functions as F
import pandas as pd
import plotly.express as px
import numpy as np
import ipywidgets as widgets
from IPython.display import display

total_anomalies = hourly_calls.filter(hourly_calls.is_anomaly == True).count()
total_rows = hourly_calls.count()
percent_anomalies = (total_anomalies / total_rows) * 100
print(f"{total_anomalies} anomalies detected ({percent_anomalies:.2f}% of all hourly records)")


means = (hourly_calls.groupBy("is_anomaly").agg(F.mean("avg_service_time").alias("mean_service_time")).toPandas())
print("\nMean average service time by anomaly flag:")
print(means)

hourly_pd = hourly_calls.toPandas()
hourly_pd["datetime"] = pd.to_datetime(hourly_pd["date"].astype(str)) + pd.to_timedelta(hourly_pd["hour"], unit="h")
hourly_pd["is_anomaly"] = hourly_pd["is_anomaly"].apply(lambda x: True if str(x).lower() in ["true", "1"] else False)
hourly_pd["status_label"] = np.where(hourly_pd["is_anomaly"], "Anomaly", "Normal")

print(hourly_pd["is_anomaly"].value_counts(dropna=False))
print(hourly_pd["status_label"].unique())


#@widgets.interact(call_type=sorted(hourly_pd['call_type'].unique()))
#def plot_by_call_type(call_type):
    #subset = hourly_pd[hourly_pd['call_type'] == call_type]
    #fig = px.box(
    #    subset,
    #    x='hour',
    #    y='avg_service_time',
    #    color='status_label',
    #    title=f'{call_type} — Response Time by Hour of Day (Anomalies Highlighted)',
    #    labels={'hour_of_day': 'Hour of Day', 'avg_service_time': 'Avg Response Time (s)', 'status_label': 'Status'},
    #)
    #fig.update_layout(template='plotly_white', title_x=0.5)
    #fig.show()


1126 anomalies detected (1.88% of all hourly records)



Mean average service time by anomaly flag:
   is_anomaly  mean_service_time
0           1        2019.561440
1           0        2221.165251
is_anomaly
False    58813
True      1126
Name: count, dtype: int64
['Normal' 'Anomaly']


The plots suggest that the model is successfully identifying outliers in response times, with clear deviations visible in several call types.
However, some anomalies appear less distinct from normal values, likely because the detection also accounts for dispatch area—so what seems typical overall may still be unusual within that specific zone’s context.


*add something to show per dispatch area?*

**Validating Call Type Mismatch Anomalies**

Most call type mismatches identified by the model represent minor wording or categorization differences rather than true labeling errors. This suggests the model is functioning correctly but that most detected mismatches are operationally routine, not anomalies of concern therfore there is no true need of validation for this type of anomaly detection.

**Validating Geographic Outlier Detection**

In [66]:
cols = ["dispatch_sector", "dispatch_latitude", "dispatch_longitude", "spatial_score", "is_geo_anomaly"]
geo_pd = (df_geo.select(*cols).sample(fraction=0.1, seed=42).toPandas()) #10 % of sample to make it grphable

print(f"Loaded {len(geo_pd):,} rows for validation sample")
geo_pd["status_label"] = geo_pd["is_geo_anomaly"].fillna(0).astype(int).map({0: "Normal", 1: "Anomaly"})

fig = px.box(
    geo_pd,
    x="status_label",
    y="spatial_score",
    color="status_label",
    title="Spatial Score Distribution (Normal vs Geographic Anomalies)",
    labels={"status_label": "Status", "spatial_score": "Spatial Score"},
)
#fig.update_layout(template="plotly_white", title_x=0.5, showlegend=False)
#fig.show()

Loaded 58,258 rows for validation sample


There is a clear separation between the normal vs the anamly box plots showing that they were indeed anomalies

**Validating Temporal Burst Detection**

In [82]:
hourly_pd = df_w_cad_hourly.select("event_hour", "total_calls", "rolling_mean", "z_score", "is_burst").toPandas()

hourly_pd = hourly_pd.sort_values("event_hour")
hourly_pd["status_label"] = hourly_pd["is_burst"].fillna(0).astype(int).map({0: "Normal", 1: "Burst"})

print("Burst status counts:")
print(hourly_pd["status_label"].value_counts(dropna=False))
print(f"Burst anomaly rate: {hourly_pd['is_burst'].mean():.2%}")

fig = px.line(
    hourly_pd,
    x="event_hour",
    y="total_calls",
    color="status_label",
    color_discrete_map={"Normal": "blue", "Burst": "red"},
    title="Temporal Burst Detection — Citywide Call Volume",
    labels={"event_hour": "Hour", "total_calls": "Total 911 Calls"},
)
#fig.update_traces(mode="lines+markers")
#fig.update_layout(template="plotly_white", title_x=0.5)
#fig.show()

burst_points = hourly_pd.query("is_burst == 1")
fig2 = px.scatter(
    burst_points,
    x="event_hour",
    y="total_calls",
    color="z_score",
    color_continuous_scale="Reds",
    title="Detected Citywide Burst Events (Z-score Intensity)",
    hover_data=["rolling_mean", "z_score"],
)
#fig2.update_layout(template="plotly_white", title_x=0.5)
#fig2.show()

Burst status counts:
status_label
Normal    2026309
Burst        6887
Name: count, dtype: int64
Burst anomaly rate: 0.34%


The z-score–based burst detection identified 6,887 burst hours out of roughly 2 million total hours, yielding an anomaly rate of only 0.34 % — a healthy, conservative signal that the model isn’t over-flagging.

Visually, the red “Burst” points align with distinct spikes in hourly 911 call volume, confirming that detections correspond to true surges rather than noise. The second chart further supports this: bursts coincide with higher z-scores (≈ 2.5–3.2 +), showing that flagged events represent statistically significant deviations from typical call rates.

**Validating Dispatch Routing Anomolies**

In [80]:
dist_pd = df_dist.select("dispatch_sector", "distance_km", "z_score", "is_routing_anomaly").toPandas()

dist_pd = dist_pd.dropna(subset=["distance_km"])
dist_pd["status_label"] = dist_pd["is_routing_anomaly"].fillna(0).astype(int).map({0: "Normal", 1: "Anomaly"})

print("Routing Anomaly Counts:")
print(dist_pd["status_label"].value_counts(dropna=False))
print(f"Routing anomaly rate: {dist_pd['is_routing_anomaly'].mean():.2%}")

# overall
fig1 = px.box(
    dist_pd,
    x="status_label",
    y="distance_km",
    color="status_label",
    color_discrete_map={"Normal": "blue", "Anomaly": "red"},
    title="Dispatch Routing Anomalies — Distribution of Travel Distance",
    labels={"status_label": "Status", "distance_km": "Travel Distance (km)"}
)
fig1.update_layout(template="plotly_white", title_x=0.5)
#fig1.show()

# by dispatch sector
fig2 = px.box(
    dist_pd,
    x="dispatch_sector",
    y="distance_km",
    color="status_label",
    color_discrete_map={"Normal": "blue", "Anomaly": "red"},
    title="Dispatch Sector vs Travel Distance (Highlighting Routing Anomalies)",
    labels={"dispatch_sector": "Dispatch Sector", "distance_km": "Travel Distance (km)"}
)
#fig2.update_layout(template="plotly_white", title_x=0.5, height=700)
#fig2.show()

#Outlier Scatter
fig3 = px.scatter(
    dist_pd,
    x="z_score",
    y="distance_km",
    color="status_label",
    color_discrete_map={"Normal": "blue", "Anomaly": "red"},
    title="Z-Score vs Travel Distance (Routing Anomalies)",
    labels={"z_score": "Distance Z-Score", "distance_km": "Travel Distance (km)"}
)
#fig3.update_layout(template="plotly_white", title_x=0.5)
#fig3.show()


Routing Anomaly Counts:
status_label
Normal     446148
Anomaly     17477
Name: count, dtype: int64
Routing anomaly rate: 3.77%


Approximately 3.8% of records were flagged as routing anomalies, representing trips with distances substantially greater than typical operational ranges. These outliers suggest the model is effectively identifying extreme or irregular routing patterns. The concentration of these anomalies at unusually high distance values may also indicate potential issues within the CAD system’s distance calculation or data pipeline, warranting further investigation.